# Padding and Stride
:label:`sec_padding`

Recall the example of a convolution in :numref:`fig_correlation`. 
The input had both a height and width of 3
and the convolution kernel had both a height and width of 2,
yielding an output representation with dimension $2\times2$.
Assuming that the input shape is $n_\textrm{h}\times n_\textrm{w}$
and the convolution kernel shape is $k_\textrm{h}\times k_\textrm{w}$,
the output shape will be $(n_\textrm{h}-k_\textrm{h}+1) \times (n_\textrm{w}-k_\textrm{w}+1)$: 
we can only shift the convolution kernel so far until it runs out
of pixels to apply the convolution to. 

In the following we will explore a number of techniques, 
including padding and strided convolutions,
that offer more control over the size of the output. 
As motivation, note that since kernels generally
have width and height greater than $1$,
after applying many successive convolutions,
we tend to wind up with outputs that are
considerably smaller than our input.
If we start with a $240 \times 240$ pixel image,
ten layers of $5 \times 5$ convolutions
reduce the image to $200 \times 200$ pixels,
slicing off $30 \%$ of the image and with it
obliterating any interesting information
on the boundaries of the original image.
*Padding* is the most popular tool for handling this issue.
In other cases, we may want to reduce the dimensionality drastically,
e.g., if we find the original input resolution to be unwieldy.
*Strided convolutions* are a popular technique that can help in these instances.


In [1]:
import torch
from torch import nn

## Padding

As described above, one tricky issue when applying convolutional layers
is that we tend to lose pixels on the perimeter of our image. Consider :numref:`img_conv_reuse` that depicts the pixel utilization as a function of the convolution kernel size and the position within the image. The pixels in the corners are hardly used at all. 

![Pixel utilization for convolutions of size $1 \times 1$, $2 \times 2$, and $3 \times 3$ respectively.](../img/conv-reuse.svg)
:label:`img_conv_reuse`

Since we typically use small kernels,
for any given convolution
we might only lose a few pixels
but this can add up as we apply
many successive convolutional layers.
One straightforward solution to this problem
is to add extra pixels of filler around the boundary of our input image,
thus increasing the effective size of the image.
Typically, we set the values of the extra pixels to zero.
In :numref:`img_conv_pad`, we pad a $3 \times 3$ input,
increasing its size to $5 \times 5$.
The corresponding output then increases to a $4 \times 4$ matrix.
The shaded portions are the first output element as well as the input and kernel tensor elements used for the output computation: $0\times0+0\times1+0\times2+0\times3=0$.

![Two-dimensional cross-correlation with padding.](../img/conv-pad.svg)
:label:`img_conv_pad`

In general, if we add a total of $p_\textrm{h}$ rows of padding
(roughly half on top and half on bottom)
and a total of $p_\textrm{w}$ columns of padding
(roughly half on the left and half on the right),
the output shape will be

$$(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+1)\times(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+1).$$

This means that the height and width of the output
will increase by $p_\textrm{h}$ and $p_\textrm{w}$, respectively.

In many cases, we will want to set $p_\textrm{h}=k_\textrm{h}-1$ and $p_\textrm{w}=k_\textrm{w}-1$
to give the input and output the same height and width.
This will make it easier to predict the output shape of each layer
when constructing the network.
Assuming that $k_\textrm{h}$ is odd here,
we will pad $p_\textrm{h}/2$ rows on both sides of the height.
If $k_\textrm{h}$ is even, one possibility is to
pad $\lceil p_\textrm{h}/2\rceil$ rows on the top of the input
and $\lfloor p_\textrm{h}/2\rfloor$ rows on the bottom.
We will pad both sides of the width in the same way.

CNNs commonly use convolution kernels
with odd height and width values, such as 1, 3, 5, or 7.
Choosing odd kernel sizes has the benefit
that we can preserve the dimensionality
while padding with the same number of rows on top and bottom,
and the same number of columns on left and right.

Moreover, this practice of using odd kernels
and padding to precisely preserve dimensionality
offers a clerical benefit.
For any two-dimensional tensor `X`,
when the kernel's size is odd
and the number of padding rows and columns
on all sides are the same,
thereby producing an output with the same height and width as the input,
we know that the output `Y[i, j]` is calculated
by cross-correlation of the input and convolution kernel
with the window centered on `X[i, j]`.

In the following example, we create a two-dimensional convolutional layer
with a height and width of 3
and (**apply 1 pixel of padding on all sides.**)
Given an input with a height and width of 8,
we find that the height and width of the output is also 8.


In [2]:
# We define a helper function to calculate convolutions. It initializes the
# convolutional layer weights and performs corresponding dimensionality
# elevations and reductions on the input and output
def comp_conv2d(conv2d, X):
    # (1, 1) indicates that batch size and the number of channels are both 1
    X = X.reshape((1, 1) + X.shape)
    Y = conv2d(X)
    # Strip the first two dimensions: examples and channels
    return Y.reshape(Y.shape[2:])

# 1 row and column is padded on either side, so a total of 2 rows or columns
# are added
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1)
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

When the height and width of the convolution kernel are different,
we can make the output and input have the same height and width
by [**setting different padding numbers for height and width.**]


In [3]:
# We use a convolution kernel with height 5 and width 3. The padding on either
# side of the height and width are 2 and 1, respectively
conv2d = nn.LazyConv2d(1, kernel_size=(5, 3), padding=(2, 1))
comp_conv2d(conv2d, X).shape

torch.Size([8, 8])

## Stride

When computing the cross-correlation,
we start with the convolution window
at the upper-left corner of the input tensor,
and then slide it over all locations both down and to the right.
In the previous examples, we defaulted to sliding one element at a time.
However, sometimes, either for computational efficiency
or because we wish to downsample,
we move our window more than one element at a time,
skipping the intermediate locations. This is particularly useful if the convolution 
kernel is large since it captures a large area of the underlying image.

We refer to the number of rows and columns traversed per slide as *stride*.
So far, we have used strides of 1, both for height and width.
Sometimes, we may want to use a larger stride.
:numref:`img_conv_stride` shows a two-dimensional cross-correlation operation
with a stride of 3 vertically and 2 horizontally.
The shaded portions are the output elements as well as the input and kernel tensor elements used for the output computation: $0\times0+0\times1+1\times2+2\times3=8$, $0\times0+6\times1+0\times2+0\times3=6$.
We can see that when the second element of the first column is generated,
the convolution window slides down three rows.
The convolution window slides two columns to the right
when the second element of the first row is generated.
When the convolution window continues to slide two columns to the right on the input,
there is no output because the input element cannot fill the window
(unless we add another column of padding).

![Cross-correlation with strides of 3 and 2 for height and width, respectively.](../img/conv-stride.svg)
:label:`img_conv_stride`

In general, when the stride for the height is $s_\textrm{h}$
and the stride for the width is $s_\textrm{w}$, the output shape is

$$\lfloor(n_\textrm{h}-k_\textrm{h}+p_\textrm{h}+s_\textrm{h})/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}-k_\textrm{w}+p_\textrm{w}+s_\textrm{w})/s_\textrm{w}\rfloor.$$

If we set $p_\textrm{h}=k_\textrm{h}-1$ and $p_\textrm{w}=k_\textrm{w}-1$,
then the output shape can be simplified to
$\lfloor(n_\textrm{h}+s_\textrm{h}-1)/s_\textrm{h}\rfloor \times \lfloor(n_\textrm{w}+s_\textrm{w}-1)/s_\textrm{w}\rfloor$.
Going a step further, if the input height and width
are divisible by the strides on the height and width,
then the output shape will be $(n_\textrm{h}/s_\textrm{h}) \times (n_\textrm{w}/s_\textrm{w})$.

Below, we [**set the strides on both the height and width to 2**],
thus halving the input height and width.


In [4]:
conv2d = nn.LazyConv2d(1, kernel_size=3, padding=1, stride=2)
comp_conv2d(conv2d, X).shape

torch.Size([4, 4])

Let's look at (**a slightly more complicated example**).


In [5]:
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
comp_conv2d(conv2d, X).shape

torch.Size([2, 2])

## Summary and Discussion

Padding can increase the height and width of the output. This is often used to give the output the same height and width as the input to avoid undesirable shrinkage of the output. Moreover, it ensures that all pixels are used equally frequently. Typically we pick symmetric padding on both sides of the input height and width. In this case we refer to $(p_\textrm{h}, p_\textrm{w})$ padding. Most commonly we set $p_\textrm{h} = p_\textrm{w}$, in which case we simply state that we choose padding $p$. 

A similar convention applies to strides. When horizontal stride $s_\textrm{h}$ and vertical stride $s_\textrm{w}$ match, we simply talk about stride $s$. The stride can reduce the resolution of the output, for example reducing the height and width of the output to only $1/n$ of the height and width of the input for $n > 1$. By default, the padding is 0 and the stride is 1. 

So far all padding that we discussed simply extended images with zeros. This has significant computational benefit since it is trivial to accomplish. Moreover, operators can be engineered to take advantage of this padding implicitly without the need to allocate additional memory. At the same time, it allows CNNs to encode implicit position information within an image, simply by learning where the "whitespace" is. There are many alternatives to zero-padding. :citet:`Alsallakh.Kokhlikyan.Miglani.ea.2020` provided an extensive overview of those (albeit without a clear case for when to use nonzero paddings unless artifacts occur). 


## Exercises

1. Given the final code example in this section with kernel size $(3, 5)$, padding $(0, 1)$, and stride $(3, 4)$, 
   calculate the output shape to check if it is consistent with the experimental result.
1. For audio signals, what does a stride of 2 correspond to?
1. Implement mirror padding, i.e., padding where the border values are simply mirrored to extend tensors. 
1. What are the computational benefits of a stride larger than 1?
1. What might be statistical benefits of a stride larger than 1?
1. How would you implement a stride of $\frac{1}{2}$? What does it correspond to? When would this be useful?


[Discussions](https://discuss.d2l.ai/t/68)



1. Given the final code example in this section with kernel size $(3, 5)$, padding $(0, 1)$, and stride $(3, 4)$, 
   calculate the output shape to check if it is consistent with the experimental result.

# Output Shape Calculation for Convolution with Given Parameters

To calculate the output shape for the convolution operation with kernel size $(3, 5)$, padding $(0, 1)$, and stride $(3, 4)$, I'll use the formula provided in the notebook:

$$\left\lfloor\frac{n_h-k_h+p_h+s_h}{s_h}\right\rfloor \times \left\lfloor\frac{n_w-k_w+p_w+s_w}{s_w}\right\rfloor$$

Where:
- $n_h, n_w$ are the input height and width
- $k_h, k_w$ are the kernel height and width
- $p_h, p_w$ are the padding for height and width
- $s_h, s_w$ are the strides for height and width

From the code example:
```python
X = torch.rand(size=(8, 8))
conv2d = nn.LazyConv2d(1, kernel_size=(3, 5), padding=(0, 1), stride=(3, 4))
```

So we have:
- Input dimensions: $n_h = 8, n_w = 8$
- Kernel size: $k_h = 3, k_w = 5$
- Padding: $p_h = 0, p_w = 1$
- Stride: $s_h = 3, s_w = 4$

Let's calculate the output height:
$$\left\lfloor\frac{8-3+0+3}{3}\right\rfloor = \left\lfloor\frac{8}{3}\right\rfloor = \lfloor 2.67 \rfloor = 2$$

And the output width:
$$\left\lfloor\frac{8-5+1+4}{4}\right\rfloor = \left\lfloor\frac{8}{4}\right\rfloor = 2$$

Therefore, the output shape should be $(2, 2)$, which matches the experimental result shown in the notebook:

```
torch.Size([2, 2])
```

This calculation confirms the theoretical understanding of how convolution output dimensions are determined by the input dimensions, kernel size, padding, and stride parameters.

To understand this intuitively: when we use a stride of 3 vertically, we can only fit the kernel twice along the height of our 8×8 input (positions 0 and 3). Similarly, with a stride of 4 horizontally and accounting for the padding of 1 on the width, we can only fit the kernel twice along the width (positions 0 and 4). This explains why we get a 2×2 output.

2. For audio signals, what does a stride of 2 correspond to?


# Stride of 2 in Audio Signal Processing

Your answer provides a starting intuition, but let's develop a more complete understanding of what a stride of 2 means for audio signals.

In audio signal processing, a stride of 2 corresponds to **downsampling the signal by a factor of 2**. Let me explain this in detail:

## Conceptual Understanding

Audio signals are one-dimensional sequences of amplitude values sampled at regular time intervals. When we apply convolution to audio:

1. The "kernel" slides across the audio signal (time domain)
2. A stride of 2 means the kernel moves 2 time steps at each slide
3. This effectively samples only every other position in the original signal

## Key Effects

A stride of 2 in audio processing:

- **Reduces the temporal resolution** by half
- **Halves the output signal length** compared to the input
- **Decreases the sampling rate** by a factor of 2
- **Reduces the Nyquist frequency** (highest frequency that can be represented) by half

## Practical Implications

- **Efficient processing**: Computing convolutions on fewer positions reduces computational cost
- **Dimensionality reduction**: Useful for extracting features at multiple temporal scales
- **Feature hierarchy**: In deep networks, helps build representations that capture longer-term patterns
- **Caution needed**: May cause aliasing if the signal contains frequencies above the new Nyquist limit

## Audio-Specific Analogy

Think of it like listening to every other beat of a song - you're still getting the general melody but with less temporal detail. The resulting output has half the temporal resolution but can still capture the important characteristics of the signal.

In practical audio applications like speech recognition or music classification, strides help create multi-scale representations that capture both fine-grained details and broader temporal patterns in the signal.

3. Implement mirror padding, i.e., padding where the border values are simply mirrored to extend tensors. 


# Implementing Mirror Padding for Tensors

Your approach has the right idea, but let's implement a complete solution using PyTorch. Mirror padding (also called "reflect padding") extends a tensor by reflecting values across the borders.

```python
def mirror_pad(X, pad_h=1, pad_w=1):
    """
    Apply mirror padding to a 2D tensor.
    
    Args:
        X: Input tensor of shape (h, w) or (batch_size, channels, h, w)
        pad_h: Padding size for height dimension (top and bottom)
        pad_w: Padding size for width dimension (left and right)
        
    Returns:
        Padded tensor with mirrored border values
    """
    # Handle different input formats
    if X.dim() == 2:
        # For a simple 2D tensor
        h, w = X.shape
        # Create output tensor with padded dimensions
        padded = torch.zeros((h + 2*pad_h, w + 2*pad_w), dtype=X.dtype, device=X.device)
        # Fill the center with the original tensor
        padded[pad_h:pad_h+h, pad_w:pad_w+w] = X
        
        # Mirror padding for top and bottom
        for i in range(pad_h):
            padded[pad_h-1-i, pad_w:pad_w+w] = X[i, :]  # Top padding (mirror)
            padded[pad_h+h+i, pad_w:pad_w+w] = X[h-1-i, :]  # Bottom padding (mirror)
        
        # Mirror padding for left and right
        for j in range(pad_w):
            padded[:, pad_w-1-j] = padded[:, pad_w+j]  # Left padding (mirror)
            padded[:, pad_w+w+j] = padded[:, pad_w+w-1-j]  # Right padding (mirror)
            
        # Corner cases - mirror from the already padded areas
        for i in range(pad_h):
            for j in range(pad_w):
                padded[pad_h-1-i, pad_w-1-j] = padded[pad_h+i, pad_w+j]  # Top-left
                padded[pad_h-1-i, pad_w+w+j] = padded[pad_h+i, pad_w+w-1-j]  # Top-right
                padded[pad_h+h+i, pad_w-1-j] = padded[pad_h+h-1-i, pad_w+j]  # Bottom-left
                padded[pad_h+h+i, pad_w+w+j] = padded[pad_h+h-1-i, pad_w+w-1-j]  # Bottom-right
                
    else:
        # For batched tensors with channels
        # Use PyTorch's built-in reflect padding which is more efficient
        return torch.nn.functional.pad(X, (pad_w, pad_w, pad_h, pad_h), mode='reflect')
    
    return padded

# Example usage
def test_mirror_padding():
    # Create a simple test tensor
    X = torch.tensor([[1, 2, 3], 
                      [4, 5, 6], 
                      [7, 8, 9]], dtype=torch.float32)
    
    # Apply mirror padding
    padded = mirror_pad(X, pad_h=2, pad_w=2)
    
    print("Original tensor:")
    print(X)
    print("\nPadded tensor with mirror padding:")
    print(padded)
    
    # Test with PyTorch's built-in function for comparison
    pytorch_padded = torch.nn.functional.pad(X.unsqueeze(0).unsqueeze(0), 
                                           (2, 2, 2, 2), mode='reflect')
    print("\nPyTorch's built-in reflect padding:")
    print(pytorch_padded.squeeze())
    
    return padded

# Run the test
test_result = test_mirror_padding()
```

## Explanation

1. **For Simple 2D Tensors**: Our implementation manually creates the padded tensor by:
   - First copying the original tensor to the center
   - Reflecting values across the top and bottom borders
   - Reflecting values across the left and right borders
   - Handling the corners by mirroring diagonally

2. **For Batched Tensors**: We leverage PyTorch's built-in `torch.nn.functional.pad()` with `mode='reflect'`, which is optimized for this exact purpose.

3. **Visual Intuition**: Mirror padding for a 3×3 tensor with `pad_h=pad_w=2` would look like:
   ```
   9 8 7 8 9
   6 5 4 5 6
   3 2 1 2 3
   6 5 4 5 6
   9 8 7 8 9
   ```

This implementation preserves edge continuity, which is often beneficial in convolutional networks as it reduces edge artifacts that can occur with zero padding.

4. What are the computational benefits of a stride larger than 1?


# Computational Benefits of Larger Strides in Convolutions

Your answer identifies the core benefit, but let's explore this in more depth to understand the significant computational advantages of using strides larger than 1.

## Primary Computational Benefits

### 1. Reduced Computation Volume
- **Fewer output elements**: With stride s > 1, the output has approximately 1/s² fewer elements in 2D convolutions
- **Fewer multiply-accumulate operations**: Each output element requires k_h × k_w multiply-accumulates, so fewer outputs mean fewer total operations

### 2. Memory Efficiency
- **Smaller activation maps**: Reduced output dimensions mean smaller feature maps throughout the network
- **Lower memory footprint**: Critical for training large models or processing high-resolution inputs
- **Better cache utilization**: Smaller tensors lead to fewer cache misses

### 3. Training and Inference Speed
- **Faster forward pass**: Fewer computations lead directly to faster processing
- **Faster backward pass**: Gradient computation is also proportionally faster
- **Lower energy consumption**: Particularly important for mobile and edge devices

## Quantitative Example

Consider a convolutional layer with a 3×3 kernel applied to a 256×256 input:

| Stride | Output Size | Relative Computation |
|--------|-------------|----------------------|
| 1      | 254×254     | 100%                 |
| 2      | 127×127     | ~25%                 |
| 4      | 63×63       | ~6.25%               |

With stride=2, we perform only about 25% of the computations compared to stride=1, while with stride=4, it drops to around 6.25%.

## Trade-off Consideration

The computational savings come at a cost of potentially losing spatial information. This trade-off is often acceptable in tasks where:

1. Downsampling is desirable for hierarchical feature learning
2. Global context matters more than local details
3. Input redundancy is high (adjacent pixels often contain similar information)

In deep neural networks, particularly in early layers of classification networks, this trade-off enables building much deeper architectures with reasonable computational requirements.

5. What might be statistical benefits of a stride larger than 1?



# Statistical Benefits of Larger Strides in Convolutions

Using a stride larger than 1 offers several important statistical benefits beyond just computational efficiency:

## Primary Statistical Benefits

### 1. Increased Receptive Field
- **Wider context capture**: Each layer's outputs incorporate information from a larger portion of the original input
- **Hierarchical feature learning**: Higher layers can more easily capture global patterns without requiring excessive depth

### 2. Reduced Overfitting
- **Parameter efficiency**: Smaller activation maps lead to fewer parameters in subsequent layers
- **Natural regularization**: Downsampling acts as a form of information bottleneck, forcing the network to learn more robust features
- **Reduced sensitivity to small translations**: The model becomes less likely to memorize exact pixel locations

### 3. Feature Diversification
- **Breaks spatial correlation**: Adjacent pixels in images are highly correlated; strided convolutions help reduce this redundancy
- **Scale invariance**: Features learned across downsampled representations help recognize patterns at different scales
- **Improved feature hierarchy**: Creates a more effective progressive abstraction from low-level to high-level features

### 4. Signal-to-Noise Enhancement
- **Noise reduction**: Acts similarly to pooling by smoothing out local variations
- **Focus on dominant features**: Helps eliminate minor variations while preserving important structural information
- **Statistical averaging**: Each feature in the downsampled representation effectively aggregates information from a larger input region

## Concrete Example

In image classification tasks, early CNN layers with stride > 1 help extract basic visual elements while discarding pixel-precise positioning. This creates a more robust statistical representation that:

1. Is less sensitive to small shifts, rotations, or deformations in the input
2. Emphasizes pattern recognition over exact spatial memorization
3. Creates a natural information hierarchy from detailed local features to broader semantic concepts

This is why architectures like ResNet and EfficientNet strategically place strided convolutions throughout their networks, creating a deliberate progression of feature abstractions that leads to better generalization on unseen data.


6. How would you implement a stride of $\frac{1}{2}$? What does it correspond to? When would this be useful?


# Implementing and Understanding Stride of $\frac{1}{2}$

Your intuition about doubling the kernel size is interesting, but it doesn't quite capture what a stride of $\frac{1}{2}$ represents. Let me explain the correct approach and its applications.

## Implementing a Stride of $\frac{1}{2}$

A stride of $\frac{1}{2}$ corresponds to **upsampling** - the opposite of what a stride > 1 does. Here's how to implement it:

### Method 1: Transposed Convolution (Deconvolution)
```python
# Using PyTorch's transposed convolution with stride=2
upsampling_layer = nn.ConvTranspose2d(
    in_channels=64,
    out_channels=64,
    kernel_size=3,
    stride=2,
    padding=1,
    output_padding=1  # Needed to handle odd dimensions correctly
)
```

### Method 2: Interpolation + Convolution
```python
def stride_half(x, kernel):
    # Step 1: Upsample the input (double dimensions)
    upsampled = torch.nn.functional.interpolate(x, scale_factor=2, mode='nearest')
    # or mode='bilinear' for smoother results
    
    # Step 2: Apply standard convolution with stride=1
    result = torch.nn.functional.conv2d(upsampled, kernel, stride=1, padding=1)
    return result
```

## What It Corresponds To

A stride of $\frac{1}{2}$ corresponds to:
1. **Spatial upsampling**: Increasing the spatial dimensions of feature maps
2. **Detail enhancement**: Generating higher-resolution representations
3. **Learning to insert information** between existing elements

Conceptually, for each step of size 1 in the original input, the convolution kernel moves by $\frac{1}{2}$ steps in the output space, creating two output elements for each input element.

## When Is This Useful?

Stride of $\frac{1}{2}$ (or transposed convolutions) is valuable in:

1. **Image super-resolution**: Enhancing low-resolution images to higher resolution
2. **Semantic segmentation**: In encoder-decoder architectures (like U-Net) where spatial information must be recovered
3. **Generative models**: GANs and VAEs use these to generate high-resolution outputs from lower-dimensional latent spaces
4. **Feature visualization**: Expanding activation maps to understand network behavior
5. **Multi-scale feature fusion**: When combining features from different network layers

## Practical Example

In image generation tasks (like image-to-image translation), you might have:
```
Input → [Encoding layers with stride>1] → Latent representation → [Decoding layers with stride=1/2] → Output
```

The "stride=1/2" operations in the decoding path gradually restore the spatial dimensions that were reduced during encoding, while learning to fill in the appropriate details.

This bidirectional dimension manipulation (downsampling then upsampling) is fundamental to many modern deep learning architectures, particularly in tasks requiring detailed spatial outputs.